# Phase 2: Supervised Fine-Tuning - THE MASTER VERSION

**CE L'ABBIAMO FATTA! I RISULTATI ORA SONO VERI E SCIENTIFICI.**

Analizzando i tuoi ultimi log:
- **79.17% di Accuratezza** (Epoca 6): Questo e un risultato **ECCELLENTE** per la predizione su persone mai viste prima (cross-subject). 
- **La curva 'ballerina'**: Dopo l'epoca 6 il modello inizia a imparare a memoria le persone dell'addestramento (Overfitting). 

**Cosa facciamo in questa versione finale (The Master):**
1. **Standardizzazione**: Ora il codice calcola la media e la deviazione standard del tuo dataset. Questo aiuta il modello a non 'spaventarsi' per valori troppo grandi o piccoli.
2. **Addestramento Dolce**: Abbassiamo il Learning Rate a `0.0001`. Il modello imparera piu lentamente ma in modo piu stabile.
3. **Risultato Finale**: Alla fine faremo un test sui soggetti che il modello non ha mai visto per avere il numero definitivo per la tua tesi.

**Riesegui tutto dalla Cella 1. Questa e la versione che 'pulisce' i risultati per la consegna.**

In [ ]:
import os, sys, shutil, re

WORK_DIR = '/kaggle/working/e4selflearning'
os.chdir('/kaggle/working')

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

print('Cloning repository...')
!git clone https://github.com/april-tools/e4selflearning.git {WORK_DIR}

print('Installing dependencies...')
!pip uninstall -y -q numpy scipy pandas
!pip install --no-cache-dir -q 'numpy==1.26.4' 'scipy==1.12.0' 'pandas<2.2' biopsykit flirt ruamel.yaml biosppy hrv-analysis neurokit2 distinctipy joypy peakutils lightning torchmetrics

print('Patching dataset.py for Standardization...')
dataset_p = os.path.join(WORK_DIR, 'src/timebase/data/dataset.py')
with open(dataset_p, 'r') as f: c = f.read()

# FIX PARENTESI: Corretto il numero di parentesi nella sostituzione
c = c.replace('return int(os.path.basename(os.path.dirname(filename)))', 'return os.path.basename(os.path.dirname(filename))')
c = c.replace('return int(os.path.basename(filename).replace(".h5", ""))', 'return os.path.basename(filename).replace(".h5", "")')
c = c.replace('data = {c: h5.get(filename, name=c).astype(np.float32) for c in self.channels}', 'data = {c: torch.from_numpy(h5.get(filename, name=c).astype(np.float32)) for c in self.channels}')

target_block = """        target = np.array(
            1 if label["status"] in self.positive_classes else 0,
            dtype=np.float32,
            ndmin=1,
        )"""
target_replacement = """        target = torch.tensor([1 if float(label["status"]) == 1.0 else 0], dtype=torch.float32)"""
c = c.replace(target_block, target_replacement)
c = c.replace('sample["subject_id"] = self.subject_ids[idx]', 'sample["subject_id"] = torch.tensor(self.subject_ids[idx], dtype=torch.float32)')

with open(dataset_p, 'w') as f: f.write(c)

print('Updating reader.py for SCALE MODE 2...')
reader_p = os.path.join(WORK_DIR, 'src/timebase/data/reader.py')
with open(reader_p, 'w') as f:
    f.write('''
import os
import pickle
import numpy as np
import typing as t
from torch.utils.data import DataLoader
from timebase.data.dataset import ClassificationDataset

def get_datasets(args, summary=None):
    meta_path = os.path.join(args.dataset, 'metadata.pkl')
    with open(meta_path, 'rb') as f: metadata = pickle.load(f)
    
    args.ds_info = metadata.get('ds_info', {})
    if 'segment_length' not in args.ds_info: args.ds_info['segment_length'] = getattr(args, 'segment_length', 60)
    args.ds_info['channel_freq'] = {'BVP': 64, 'EDA': 4, 'TEMP': 4, 'ACC_x': 32, 'ACC_y': 32, 'ACC_z': 32}
    args.input_shapes = {c: [int(args.ds_info['segment_length'] * args.ds_info['channel_freq'].get(c, 32))] for c in args.ds_info['channel_freq'].keys()}
    
    paths = metadata['sessions_paths']
    labels = metadata['sessions_labels']
    sub_ids = labels['Sub_ID']
    u_subs = np.unique(sub_ids)
    np.random.seed(args.seed)
    np.random.shuffle(u_subs)
    tr_s = u_subs[:int(0.7 * len(u_subs))]
    args.num_train_subjects = len(tr_s)
    
    defaults = {'scaling_mode': 2, 'representation_module': 'resnet', 'encoder_type': 'resnet', 'emb_num_filters': 4, 'emb_dim': 64, 'num_blocks': 3, 'num_heads': 3, 'num_units': 64}
    for k, v in defaults.items():
        if not hasattr(args, k): setattr(args, k, v)
    
    def split_data(subs):
        m = np.isin(sub_ids, subs)
        return paths[m], {k: v[m] for k, v in labels.items()}, sub_ids[m]
    
    x_tr, y_tr, sid_tr = split_data(tr_s)
    x_val, y_val, sid_val = split_data(u_subs[len(tr_s):len(tr_s)+int(0.15*len(u_subs))])
    x_te, y_te, sid_te = split_data(u_subs[len(tr_s)+int(0.15*len(u_subs)):])
    
    stats_path = os.path.join(args.dataset, 'stats.pkl')
    if os.path.exists(stats_path):
        with open(stats_path, 'rb') as f: stats = pickle.load(f)
    else: stats = metadata.get('stats', {c: {'min':-100, 'max':100, 'mean':0, 'std':1} for c in args.ds_info['channel_freq'].keys()})
    
    str2num = {s: i for i, s in enumerate(u_subs)}
    dataloader_kwargs = {'batch_size': args.batch_size, 'num_workers': 1, 'pin_memory': True}
    
    return (
        DataLoader(ClassificationDataset(args, x_tr, y_tr, sid_tr, stats, str2num, sid_tr), shuffle=True, **dataloader_kwargs),
        DataLoader(ClassificationDataset(args, x_val, y_val, sid_val, stats, str2num, sid_val), shuffle=False, **dataloader_kwargs),
        DataLoader(ClassificationDataset(args, x_te, y_te, sid_te, stats, str2num, sid_te), shuffle=False, **dataloader_kwargs)
    )
''')

train_p = os.path.join(WORK_DIR, 'train_ann.py')
with open(train_p, 'r') as f: lines = f.readlines()
with open(train_p, 'w') as f:
    for line in lines:
        if 'parser = argparse.ArgumentParser' in line:
            f.write(line)
            f.write('    parser.add_argument("--dataset", type=str, required=True)\n')
            f.write('    parser.add_argument("--segment_length", type=int, default=512)\n')
        elif '--dataset' in line or '--segment_length' in line: continue
        else: f.write(line)

print('Everything Ready. RESTART KERNEL.')
os._exit(0)


Cloning repository...
Cloning into '/kaggle/working/e4selflearning'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 71 (delta 12), reused 59 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 94.89 KiB | 2.21 MiB/s, done.
Resolving deltas: 100% (12/12), done.
Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 191.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 267.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 260.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 204.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 265.3 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37

### 1. Preprocessing & Stats Calculation

In [1]:
import os, sys, glob, shutil, pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
sys.path.append('/kaggle/working/e4selflearning/src')
from timebase.utils import h5

def sub2id(sub):
    if sub == '1': return 27.0
    return float(ord(sub.upper()) - 64)

os.chdir('/kaggle/working/e4selflearning')
BASE_DIR = 'data/preprocessed/unsegmented_indian'
if os.path.exists(BASE_DIR): shutil.rmtree(BASE_DIR)
os.makedirs(BASE_DIR, exist_ok=True)

subject_folders = sorted(glob.glob('/kaggle/input/**/subject_*', recursive=True))
dataset_dir = os.path.dirname(subject_folders[0])
sessions_info = {}; task_map = {1: 1.0, 2: 1.0, 3: 1.0, 4: 0.0, 5: 0.0}
CHANNELS = ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP', 'HR']
FREQ = {'BVP': 64, 'EDA': 4, 'TEMP': 4, 'ACC_x': 32, 'ACC_y': 32, 'ACC_z': 32, 'HR': 1}

global_data = {c: [] for c in CHANNELS}
print('Processing subjects...')
for sub_folder in tqdm(sorted([d for d in os.listdir(dataset_dir) if d.startswith('subject_')])):
    sub_path = os.path.join(dataset_dir, sub_folder); sub_name = sub_folder.replace('subject_', '')
    sub_numeric_id = sub2id(sub_name)
    for tid, status_val in task_map.items():
        task_dirs = glob.glob(os.path.join(sub_path, "**", f"T{tid}"), recursive=True)
        if not task_dirs: continue
        session_id = f"{sub_name}_T{tid}"; session_dir = os.path.join(BASE_DIR, session_id); os.makedirs(session_dir, exist_ok=True)
        data_dict = {}; max_len = 0
        for channel in CHANNELS:
            f_name = channel.split('_')[0] if 'ACC' in channel else channel
            files = glob.glob(os.path.join(task_dirs[0], f"*T{tid}{f_name}.csv")) or glob.glob(os.path.join(task_dirs[0], f"{f_name}.csv")) or glob.glob(os.path.join(task_dirs[0], f"*{f_name}*.csv"))
            if files:
                try:
                    df = pd.read_csv(files[0], header=None, skiprows=2, on_bad_lines='skip')
                    sig = df.iloc[:, {'x':0,'y':1,'z':2}[channel.split('_')[1]] if 'ACC' in channel else 0].values.astype(np.float32)
                    data_dict[channel] = sig
                    global_data[channel].append(sig)
                    max_len = max(max_len, len(sig) / FREQ.get(channel, 32))
                except: pass
        if max_len < 60: shutil.rmtree(session_dir); continue
        duration = int(max_len)
        for c in CHANNELS:
            target = duration * FREQ.get(c, 32)
            if c not in data_dict: data_dict[c] = np.zeros(target, dtype=np.float32)
            else: data_dict[c] = np.pad(data_dict[c], (0, max(0, target - len(data_dict[c]))), mode='edge')[:target]
        labels = np.zeros(41, dtype=np.float32); labels[0] = sub_numeric_id; labels[3] = status_val
        data_dict['labels'] = labels; data_dict['SLEEP'] = np.zeros(duration * 32, dtype=np.float32)
        h5.write(os.path.join(session_dir, 'channels.h5'), data_dict, overwrite=True)
        sessions_info[session_id] = {'channel_names': CHANNELS + ['IBI', 'SLEEP'], 'sampling_rates': {**FREQ, 'SLEEP': 32, 'IBI': 1}, 'unix_t0': {c: 0 for c in CHANNELS + ['IBI']}, 'Sub_ID': sub_numeric_id}

print('Calculating global statistics for standardization...')
stats = {}
for c in CHANNELS:
    if global_data[c]:
        all_v = np.concatenate(global_data[c])
        stats[c] = {'min': float(np.min(all_v)), 'max': float(np.max(all_v)), 'mean': float(np.mean(all_v)), 'std': float(np.std(all_v))}
    else: stats[c] = {'min': -100.0, 'max': 100.0, 'mean': 0.0, 'std': 1.0}

with open(os.path.join(BASE_DIR, 'metadata.pkl'), 'wb') as f: pickle.dump({'sessions_info': sessions_info, 'stats': stats}, f)
print(f'Done! Created {len(sessions_info)} sessions.')


2026-04-27 13:30:08.374643: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777296608.606803     141 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777296608.670009     141 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777296609.265278     141 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777296609.265326     141 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777296609.265329     141 computation_placer.cc:177] computation placer alr

Processing subjects...


100%|██████████| 25/25 [00:06<00:00,  4.00it/s]

Calculating global statistics for standardization...
Done! Created 59 sessions.


### 2. Segmentation

In [2]:
import os, pickle
SEG_DIR = 'data/preprocessed/sl60_ss30_indian'
print('Running segmentation...')
!PYTHONPATH=src python segment.py --data_dir data/preprocessed/unsegmented_indian --output_dir {SEG_DIR} --segment_length 60 --segmentation_mode 1 --step_size 30 --num_workers 1 --overwrite

# Copiamo le statistiche calcolate nella cella precedente
with open('data/preprocessed/unsegmented_indian/metadata.pkl', 'rb') as f: meta = pickle.load(f)
with open(os.path.join(SEG_DIR, 'stats.pkl'), 'wb') as f: pickle.dump(meta['stats'], f)
print('Segmentation & Stats Ready.')


Running segmentation...
2026-04-27 13:30:55.314318: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777296655.338719     168 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777296655.346512     168 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777296655.368211     168 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777296655.368240     168 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777296655.368244     168 computation_placer.cc:177

### 3. Fine-Tuning (Stable Version)

In [4]:
import glob, os, shutil
ckpt_files = glob.glob('/kaggle/input/**/model_state.pt', recursive=True)
STAGE_DIR = '/kaggle/working/weights/ckpt_sslearner'
os.makedirs(STAGE_DIR, exist_ok=True); shutil.copy(ckpt_files[0], os.path.join(STAGE_DIR, 'model_state.pt'))

with open(os.path.join('/kaggle/working/weights', 'args.yaml'), 'w') as f:
    f.write('''
task_mode: 1
scaling_mode: 2
pretext_task: masked_prediction
encoder_type: resnet
representation_module: resnet
mask_ratio: 0.5
split_mode: 0
include_hr: False
emb_dim: 64
emb_num_filters: 4
emb_type: 1
mlp_dim: 64
num_blocks: 3
num_heads: 3
num_units: 64
a_dropout: 0.1
m_dropout: 0.1
dropout: 0.1
drop_path: 0.1
disable_bias: False
''')

SEG_DIR = 'data/preprocessed/sl60_ss30_indian'
print('Starting Master training loop (LR=0.0001, Weight Decay=0.01, Scaling=2)...')
!PYTHONPATH=src python train_ann.py --dataset {SEG_DIR} --path2pretraining_res /kaggle/working/weights --output_dir 'output_finetune' --task_mode 1 --e4selflearning --epochs 50 --batch_size 16 --lr 0.0001 --weight_decay 0.01 --segment_length 60


Starting Master training loop (LR=0.0001, Weight Decay=0.01, Scaling=2)...
2026-04-27 13:31:33.304283: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777296693.326724     188 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777296693.335366     188 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777296693.355422     188 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777296693.355450     188 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:

In [5]:
import os, shutil, glob

# Cerchiamo automaticamente la cartella generata nella cella 3
base_path = '/kaggle/working/e4selflearning/data/preprocessed'
possible_folders = glob.glob(os.path.join(base_path, 'sl*_indian'))

if possible_folders:
    SOURCE_PATH = possible_folders[0] 
    folder_name = os.path.basename(SOURCE_PATH)
    ARCHIVE_NAME = f'/kaggle/working/{folder_name}.tar.gz'
    
    print(f"Cartella trovata: {folder_name}")
    print(f"Inizio compressione...")
    
    # Crea l'archivio
    shutil.make_archive(f'/kaggle/working/{folder_name}', 'gztar', SOURCE_PATH)
    
    print(f"✅ ARCHIVIO CREATO: {ARCHIVE_NAME}")
    print(f"Dimensione: {os.path.getsize(ARCHIVE_NAME) / (1024*1024):.2f} MB")
else:
    print("❌ ERRORE: Non ho trovato nessuna cartella di segmentazione!")


Cartella trovata: sl60_ss30_indian
Inizio compressione...
✅ ARCHIVIO CREATO: /kaggle/working/sl60_ss30_indian.tar.gz
Dimensione: 4.47 MB
